In [1]:
from bs4 import BeautifulSoup
import requests

In [50]:
# the url 
URL = "https://www.buyrentkenya.com/houses-for-sale"
URL = "https://www.buyrentkenya.com/flats-apartments-for-sale"

In [51]:
# 2. Set headers to mimic a real Ubuntu Chrome browser session
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
} 

In [52]:
try:
    response = requests.get(URL, headers=HEADERS, timeout=15)
    if response.status_code == 200:
        print("Success! Status Code 200.")
        # 4. Turn the raw text into a searchable BeautifulSoup object
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Print out the page title to verify we got the right page
        if soup.title:
            print(f"Page Title: {soup.title.text.strip()}")
        else:
            print("Connected, but couldn't find a page title tag.")
            
    else:
        print(f"Failed to fetch page. Status Code: {response.status_code}")

except Exception as e:
    print(f"An error occurred while connecting: {e}")

Success! Status Code 200.
Page Title: Apartments for Sale in Kenya | BuyRentKenya


In [53]:
soup.select_one(".pagination-page-nav")

<ul class="list-reset pagination-page-nav flex w-auto justify-center space-x-1 p-0 md:space-x-3">
<!--[if BLOCK]><![endif]--> <li class="page-item 1" key="1">
<!--[if BLOCK]><![endif]--> <p class="relative inline-flex h-10 w-10 items-center justify-center rounded-2xl bg-secondary-500 text-base font-semibold text-white hover:bg-secondary-700">
                                    1
                                </p>
<!--[if ENDBLOCK]><![endif]--> </li>
<li class="page-item" key="2">
<!--[if BLOCK]><![endif]--> <a class="bg-highlight relative inline-flex h-10 w-10 items-center justify-center rounded-2xl text-base font-semibold text-secondary hover:bg-gray-50" href="https://www.buyrentkenya.com/flats-apartments-for-sale?page=2">
                                    2
                                </a>
<!--[if ENDBLOCK]><![endif]--> </li>
<li class="page-item" key="3">
<!--[if BLOCK]><![endif]--> <a class="bg-highlight relative inline-flex h-10 w-10 items-center justify-center rounded-2x

In [54]:
# 1. Isolate the pagination container
nav = soup.select_one(".pagination-page-nav")

if nav:
    # 2. Extract all purely numeric text strings and convert to integers
    page_numbers = [int(item) for item in nav.stripped_strings if item.isdigit()]
    
    # 3. Get the last number in the sequence
    total_pages = page_numbers[-1] if page_numbers else 1
    print(f"Total pages: {total_pages}")
else:
    total_pages = 1
    print("Pagination not found, defaulting to 1 page.")


total_pages = 107

Total pages: 382


In [55]:

    def fetch_search_page(page: int = 1) -> BeautifulSoup:
        url = f"{URL}?page={page}"
        print(url)
        response = requests.get(url=url)
        return BeautifulSoup(response.text, "html.parser")

In [56]:
fetch_soup = fetch_search_page(107)

https://www.buyrentkenya.com/flats-apartments-for-sale?page=107


In [57]:
fetch_soup

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width,initial-scale=1.0,viewport-fit=cover" name="viewport"/>
<link href="https://assets.buyrentkenya.com/theme/brk/assets/apple-touch-icon-4p9S39EE.png" rel="apple-touch-icon" sizes="180x180"/>
<link href="https://assets.buyrentkenya.com/theme/brk/assets/favicon-32x32-Dz8ubMwZ.png" rel="icon" sizes="32x32" type="image/png"/>
<link href="https://assets.buyrentkenya.com/theme/brk/assets/favicon-16x16-D2LWpOaO.png" rel="icon" sizes="16x16" type="image/png"/>
<link href="https://www.buyrentkenya.com/manifest.json" rel="manifest"/>
<link color="#b72327" href="https://assets.buyrentkenya.com/theme/brk/assets/safari-pinned-tab-D633-S-9.svg" rel="mask-icon"/>
<link href="https://assets.buyrentkenya.com/theme/brk/assets/favicon-D6NWSR_P.ico" rel="shortcut icon"/>
<meta content="yes" name="apple-mobile-web-app-capable"/>
<meta content="black" name="

In [58]:
cards = fetch_soup.select(".listing-card")


In [59]:
for card in cards:
    print(card)
    break

<div class="listing-card relative mb-5 w-full md:mx-0 md:min-h-56" data-cy="listing-4009634" id="listing-4009634" x-init="Alpine.store('ga4Items').register(
        '4009634',
        JSON.parse('{\u0022item_id\u0022:\u0022LT_4009634\u0022,\u0022prev_item_id\u0022:\u0022LT_4009634\u0022,\u0022item_name\u0022:\u00223 Bed Apartment with En Suite in Kilimani\u0022,\u0022affiliation\u0022:\u0022Palmora Properties Kenya\u0022,\u0022index\u0022:6,\u0022item_brand\u0022:\u0022classified listing\u0022,\u0022item_category\u0022:\u0022Apartments for sale\u0022,\u0022item_category2\u0022:\u0022Other Apartments for sale\u0022,\u0022item_category3\u0022:\u00223 bedrooms\u0022,\u0022item_category4\u0022:\u0022not applicable\u0022,\u0022item_variant\u0022:\u0022photo only\u0022,\u0022item_seller_id\u0022:\u0022not applicable\u0022,\u0022item_new_seller_id\u0022:\u002235407\u0022,\u0022location_id\u0022:\u0022Nairobi Kilimani Kilimani\u0022,\u0022price\u0022:10000000,\u0022quantity\u0022:1,\u0022item_

In [60]:
import json
import re

def extract_listing_features(card):
    features = {}

    # 1. Base Unique Identifier
    features["listing_id"] = card.get("id", "").replace("listing-", "")

    # 2. Extract basic info from the data attributes (High Reliability)
    bi_element = card.find(attrs={"data-bi-listing-price": True})
    if bi_element:
        features["price"] = int(bi_element.get("data-bi-listing-price", 0))
        features["property_type"] = bi_element.get("data-bi-listing-category")
    else:
        features["price"] = None
        features["property_type"] = None

    # 3. Extract Bedrooms from the DOM badge
    beds_span = card.find(attrs={"data-cy": "card-bedroom_count"})
    if beds_span:
        # Extract just the number (e.g., "6 Bedrooms" -> 6)
        beds_match = re.search(r"\d+", beds_span.text)
        features["bedrooms"] = int(beds_match.group()) if beds_match else None
    else:
        features["bedrooms"] = None

    # 4. Goldmine: Extract detailed features from the embedded Alpine/GA4 JSON
    x_init_attr = card.get("x-init", "")
    # Regex to find the content inside the first JSON.parse('...')
    json_match = re.search(r"JSON\.parse\('(.*?)'\)", x_init_attr)

    if json_match:
        try:
            # Unescape the unicode quotes (\u0022 -> ")
            clean_json_str = json_match.group(1).replace(r"\u0022", '"')
            ga4_data = json.loads(clean_json_str)

            # Pull cleanly formatted ML features directly from the JSON
            features["county"] = ga4_data.get("propertyCounty")
            features["city"] = ga4_data.get("propertyCity")
            features["area"] = ga4_data.get("propertyArea")
            features["days_on_market"] = ga4_data.get(
                "propertyDaysInMarket"
            )  # Massive win for ML!
            features["seller_type"] = ga4_data.get("sellerType")

        except (json.JSONDecodeError, IndexError):
            # Fallback values if the specific card's JSON parsing fails
            features["county"] = None
            features["city"] = None
            features["area"] = None
            features["days_on_market"] = None
            features["seller_type"] = None

    return features

In [61]:
dataset = []

for card in cards:
    card_data = extract_listing_features(card)
    # Filter out records that missed a price (can't train without a target label)
    if card_data["price"]:
        dataset.append(card_data)

# Print the first extracted dictionary to verify
print(dataset[2])

{'listing_id': '4009631', 'price': 6000000, 'property_type': 'Other Apartments', 'bedrooms': 2, 'county': 'Nairobi', 'city': 'Kilimani', 'area': 'Kilimani', 'days_on_market': 6, 'seller_type': 'enterprise'}


In [62]:
data = []
for page in range(total_pages):
    page_data = fetch_search_page(page)
    cards = page_data.select(".listing-card")
    for card in cards:
        card_data = extract_listing_features(card)
        # Filter out records that missed a price (can't train without a target label)
        if card_data["price"]:
            data.append(card_data)
    if page >=2: 
        break
    
    
    

https://www.buyrentkenya.com/flats-apartments-for-sale?page=0
https://www.buyrentkenya.com/flats-apartments-for-sale?page=1
https://www.buyrentkenya.com/flats-apartments-for-sale?page=2


In [63]:
data

[{'listing_id': '4009419',
  'price': 13000000,
  'property_type': 'Other Apartments',
  'bedrooms': 4,
  'county': 'Nairobi',
  'city': 'Lavington',
  'area': 'Lavington',
  'days_on_market': 3,
  'seller_type': 'not applicable'},
 {'listing_id': '4010046',
  'price': 6000000,
  'property_type': 'Other Apartments',
  'bedrooms': 2,
  'county': 'Nairobi',
  'city': 'Kileleshwa',
  'area': 'Kileleshwa',
  'days_on_market': 3,
  'seller_type': 'not applicable'},
 {'listing_id': '4007736',
  'price': 17000000,
  'property_type': 'Other Apartments',
  'bedrooms': 3,
  'county': 'Mombasa',
  'city': 'Nyali',
  'area': 'Nyali Area',
  'days_on_market': 7,
  'seller_type': 'enterprise'},
 {'listing_id': '4009901',
  'price': 3000000,
  'property_type': 'Other Apartments',
  'bedrooms': 1,
  'county': 'Nairobi',
  'city': 'Kilimani',
  'area': 'Kilimani',
  'days_on_market': 0,
  'seller_type': 'not applicable'},
 {'listing_id': '4013139',
  'price': 5000000,
  'property_type': 'Other Apartmen